# Dados Meteorológicos

Este notebook tem duas finalidades:
1. Apresentar os metadados dos dados meteorológicos disponíveis no data lake do projeto.
1. Apresentar exemplos de acesso aos dados empregando python.

# Visão Geral

O data lake está dividido em três buckets:

1. `landing`
    * Dados brutos, sem qualquer processamento.
2. `staged`
    * Dados otimizados.
3. `curated`
    * Dados curados, prontos para serem consumidos.

Este notebook apresenta os dados meteorológicos curados, _i.e._, os dados disponíveis no bucket `curated`. 

A exceção é a seção `Informações dos Instrumentos Meteorológicos`, que carrega os dados do bucket `landing`.

# Data Lake Wrapper

In [1]:
import os
from pyarrow import fs, parquet as pq
from getpass import getpass


class DataLakeWrapper():
    
    def __init__(self) -> None:
        self._login()
        self._filesystem = self._build_filesystem()

    
    def read_parquet_dataset(self, path, filters=None):
        table = pq.ParquetDataset(
            path,
            filters=filters,
            filesystem=self._filesystem,
        ).read()
        return table
    
    
    def read_parquet_table(self, path, filters=None, columns=None):
        table = pq.read_table(
            path,
            filters=filters,
            columns=columns,
            filesystem=self._filesystem
        )
        return table

    
    def _login(self):
        os.environ['MINIO_ENDPOINT'] = input('Enter the Minio endpoint: ')
        os.environ['MINIO_USER'] = input('Enter the Minio user: ')
        os.environ['MINIO_PASSWORD'] = getpass('Enter the Minio password: ')
    

    def _build_filesystem(self):
        return fs.S3FileSystem(
            endpoint_override=os.getenv('MINIO_ENDPOINT'),
            access_key=os.getenv('MINIO_USER'),
            secret_key=os.getenv('MINIO_PASSWORD'),
            scheme='http'
        )

In [2]:
dl = DataLakeWrapper()

# Informações dos Instrumentos Meteorológicos

## Estações do Alerta Rio

In [3]:
path = 'landing/instruments_info/alertario_stations.parquet'
df = dl.read_parquet_table(path).to_pandas()
df

,id_estacao,estacao,estacao_desc,latitude,longitude,cota,x,y
0,8,Ilha do governador,ilha_do_governador,-22.81806,-43.21028,0,6.837087e+08,7.475960e+09
1,20,Guaratiba,guaratiba,-23.05028,-43.59472,0,6.439722e+08,7.450214e+06
2,16,Jardim botanico,jardim_botanico,-22.97278,-43.22389,0,6.821335e+05,7.458453e+09
3,19,Riocentro,riocentro,-22.97721,-43.39155,0,6.648794e+05,7.458100e+06
4,17,Barrinha,barrinha,-23.00849,-43.29965,7,6.742621e+08,7.454521e+09
5,30,Recreio,recreio,-23.01000,-43.44056,10,6.598168e+08,7.454514e+09
6,25,Grota funda,grota_funda,-23.01444,-43.52139,11,6.515264e+08,7.454108e+09
7,15,Saude,saude,-22.89606,-43.18786,15,6.858751e+08,7.466833e+09
8,22,Santa cruz,santa_cruz,-22.90944,-43.68444,15,6.349159e+08,7.465594e+09
9,18,Cidade de deus,cidade_de_deus,-22.94556,-43.36278,15,6.679282e+08,7.461633e+09


## Estações do INMET

In [4]:
path = 'landing/instruments_info/inmet_stations.parquet'
df = dl.read_parquet_table(path).to_pandas()
df

,cd_estacao,dc_nome,dc_nome_desc,altitude,latitude,longitude,dt_inicio_operacao
0,A602,Marambaia,marambaia,12.00,-23.050278,-43.595556,2002-11-07 22:00:00-02:00
1,A621,Vila Militar,vila_militar,30.43,-22.861389,-43.411389,2007-04-12 21:00:00-03:00
2,A652,Forte De Copacabana,forte_de_copacabana,25.59,-22.988333,-43.190556,2007-05-17 21:00:00-03:00
3,A636,Jacarepagua,jacarepagua,20.00,-22.940000,-43.402778,2017-08-09 21:00:00-03:00


## Estações do sistema Websirenes

In [5]:
path = 'landing/instruments_info/websirenes_stations.parquet'
df = dl.read_parquet_table(path).to_pandas()
df

,id_estacao,estacao,estacao_desc,latitude,longitude
0,33,Ladeira dos Tabajaras,ladeira_dos_tabajaras,-22.961700,-43.188000
1,8,Cabritos 1,cabritos_1,-22.964700,-43.195000
2,26,Guararapes 1,guararapes_1,-22.944700,-43.208000
3,34,Liberdade 1,liberdade_1,-22.926600,-43.218000
4,64,Salgueiro 1,salgueiro_1,-22.930200,-43.226000
...,...,...,...,...,...
78,5,Barão 1,barao_1,-22.902300,-43.343400
79,22,Espírito Santo 1,espirito_santo_1,-22.890000,-43.343900
80,63,Rua Quiririm 2,rua_quiririm_2,-22.888600,-43.356900
81,18,Comandante Luiz Souto 1,comandante_luiz_souto_1,-22.901700,-43.359400


# Pluviômetros do Alerta Rio

In [9]:
path = 'curated/rain_gauge/alertario'
table = dl.read_parquet_dataset(path)
table.schema

station: string
datetime: timestamp[us]
precipitation: double
hour_sin: double
hour_cos: double
latitude: double
longitude: double
year: dictionary<values=int32, indices=int32, ordered=0>
month: dictionary<values=int32, indices=int32, ordered=0>
-- schema metadata --
naming_authority: 'Alerta Rio'
timezone: 'America/Sao_Paulo'
instrument: 'rain_gauge'
variable_1: 'station - station name'
variable_2: 'latitude (degrees)'
variable_3: 'longitude (degrees)'
variable_4: 'precipitation (mm/15min)'
variable_5: 'hour_sin - sine encoding of the time of day'
variable_6: 'hour_cos - cosine encoding of the time of day'

In [10]:
df = table.to_pandas()
df

,station,datetime,precipitation,hour_sin,hour_cos,latitude,longitude,year,month
0,santa_cruz,2010-01-01 00:07:20,0.0,0.030539,0.999534,-22.90944,-43.68444,2010,1
1,santa_cruz,2010-01-01 00:22:20,0.0,0.095846,0.995396,-22.90944,-43.68444,2010,1
2,santa_cruz,2010-01-01 00:37:20,0.0,0.160743,0.986996,-22.90944,-43.68444,2010,1
3,santa_cruz,2010-01-01 00:52:20,0.0,0.224951,0.974370,-22.90944,-43.68444,2010,1
4,santa_cruz,2010-01-01 01:07:20,0.0,0.288196,0.957571,-22.90944,-43.68444,2010,1
...,...,...,...,...,...,...,...,...,...
5592054,riocentro,2014-09-30 22:45:00,0.0,-0.321439,0.946930,-22.97721,-43.39155,2014,9
5592055,riocentro,2014-09-30 23:00:00,0.0,-0.258819,0.965926,-22.97721,-43.39155,2014,9
5592056,riocentro,2014-09-30 23:15:00,0.0,-0.195090,0.980785,-22.97721,-43.39155,2014,9
5592057,riocentro,2014-09-30 23:30:00,0.0,-0.130526,0.991445,-22.97721,-43.39155,2014,9


# Estações Meteorológicas do Alerta Rio

In [11]:
path = 'curated/weather_station/alertario'
table = dl.read_parquet_dataset(path)
table.schema

station: string
datetime: timestamp[us, tz=UTC]
precipitation: double
wind_dir: double
wind_speed: double
temperature: double
pressure: double
humidity: double
wind_u: double
wind_v: double
hour_sin: double
hour_cos: double
month_sin: double
month_cos: double
latitude: double
longitude: double
year: dictionary<values=int32, indices=int32, ordered=0>
month: dictionary<values=int32, indices=int32, ordered=0>
-- schema metadata --
naming_authority: 'Alerta Rio'
timezone: 'UTC'
instrument: 'weather station'
variable_1: 'station - station name'
variable_2: 'datetime - UTC datetime of the measurement'
variable_3: 'precipitation (mm/15min)'
variable_4: 'wind_dir (degrees)'
variable_5: 'wind_speed (km/h)'
variable_6: 'temperature (degrees Celsius)'
variable_7: 'pressure (hPa)'
variable_8: 'humidity (%)'
variable_9: 'wind_u (cyclic U component from the wind)'
variable_10: 'wind_v (cyclic V component from the wind)'
variable_11: 'hour_sin (sine encoding of the time of day)'
variable_12: 'hour_co

In [12]:
df = table.to_pandas()
df

,station,datetime,precipitation,wind_dir,wind_speed,temperature,pressure,humidity,wind_u,wind_v,hour_sin,hour_cos,month_sin,month_cos,latitude,longitude,year,month
0,iraja,2016-01-01 02:00:00+00:00,0.0,NaN,NaN,30.8,NaN,60.0,NaN,NaN,0.500000,0.866025,0.500000,0.866025,-22.82694,-43.33694,2016,1
1,iraja,2016-01-01 02:15:00+00:00,0.0,NaN,NaN,30.8,NaN,59.0,NaN,NaN,0.555570,0.831470,0.500000,0.866025,-22.82694,-43.33694,2016,1
2,iraja,2016-01-01 02:30:00+00:00,0.0,NaN,NaN,30.6,NaN,60.0,NaN,NaN,0.608761,0.793353,0.500000,0.866025,-22.82694,-43.33694,2016,1
3,iraja,2016-01-01 02:45:00+00:00,0.0,NaN,NaN,30.8,NaN,60.0,NaN,NaN,0.659346,0.751840,0.500000,0.866025,-22.82694,-43.33694,2016,1
4,iraja,2016-01-01 03:00:00+00:00,0.0,NaN,NaN,30.6,NaN,61.0,NaN,NaN,0.707107,0.707107,0.500000,0.866025,-22.82694,-43.33694,2016,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1913632,guaratiba,2023-04-01 01:45:00+00:00,0.0,131.0,4.6,23.1,1008.9,92.0,-3.471664,3.017872,0.442289,0.896873,0.866025,-0.500000,-23.05028,-43.59472,2023,4
1913633,guaratiba,2023-04-01 02:00:00+00:00,0.0,109.0,9.5,22.9,1009.2,92.0,-8.982426,3.092897,0.500000,0.866025,0.866025,-0.500000,-23.05028,-43.59472,2023,4
1913634,guaratiba,2023-04-01 02:15:00+00:00,0.0,106.0,3.2,22.9,1009.2,92.0,-3.076037,0.882040,0.555570,0.831470,0.866025,-0.500000,-23.05028,-43.59472,2023,4
1913635,guaratiba,2023-04-01 02:30:00+00:00,0.0,114.0,4.2,22.8,1008.9,92.0,-3.836891,1.708294,0.608761,0.793353,0.866025,-0.500000,-23.05028,-43.59472,2023,4


# Estações Meteorológicas do INMET

In [13]:
path = 'curated/weather_station/inmet'
table = dl.read_parquet_dataset(path)
table.schema

station_id: string
datetime: timestamp[us, tz=America/Sao_Paulo]
precipitation: double
pressure: double
temperature: double
dew_point: double
humidity: double
wind_dir: double
wind_speed: double
wind_u: double
wind_v: double
hour_sin: double
hour_cos: double
station: string
latitude: double
longitude: double
year: dictionary<values=int32, indices=int32, ordered=0>
month: dictionary<values=int32, indices=int32, ordered=0>
-- schema metadata --
naming_authority: 'INMET - Instituto Nacional de Meteorologia'
timezone: 'America/Sao_Paulo'
instrument: 'weather station'
variable_1: 'station_id (station UID)'
variable_2: 'datetime (datetime of the measurement)'
variable_3: 'precipitation (hourly precipitation in mm)'
variable_4: 'pressure (instant atmospheric pressure in mB)'
variable_5: 'temperature (instant temperature in Celsius)'
variable_6: 'dew_point (dew point temperature in Celsius)'
variable_7: 'humidity (instant relative humidity in %)'
variable_8: 'wind_dir (clockwise wind direction

In [14]:
df = table.to_pandas()
df

,station_id,datetime,precipitation,pressure,temperature,dew_point,humidity,wind_dir,wind_speed,wind_u,wind_v,hour_sin,hour_cos,station,latitude,longitude,year,month
0,A602,2002-11-07 22:00:00-02:00,0.0,1021.4,18.2,15.7,86.0,28.0,2.0,-0.938943,-1.765895,-0.500000,0.866025,marambaia,-23.050278,-43.595556,2002,11
1,A602,2002-11-07 23:00:00-02:00,0.0,1021.9,18.5,16.2,87.0,348.0,2.5,0.519779,-2.445369,-0.258819,0.965926,marambaia,-23.050278,-43.595556,2002,11
2,A602,2002-11-08 00:00:00-02:00,3.6,1021.7,17.8,15.9,89.0,17.0,2.5,-0.730929,-2.390762,0.000000,1.000000,marambaia,-23.050278,-43.595556,2002,11
3,A602,2002-11-08 01:00:00-02:00,0.0,1020.9,17.4,15.7,90.0,29.0,1.8,-0.872657,-1.574315,0.258819,0.965926,marambaia,-23.050278,-43.595556,2002,11
4,A602,2002-11-08 02:00:00-02:00,0.0,1020.3,17.2,15.6,91.0,2.0,2.4,-0.083759,-2.398538,0.500000,0.866025,marambaia,-23.050278,-43.595556,2002,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533707,A602,2023-09-30 22:00:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.500000,0.866025,marambaia,-23.050278,-43.595556,2023,9
533708,A602,2023-09-30 23:00:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.258819,0.965926,marambaia,-23.050278,-43.595556,2023,9
533709,A621,2023-09-30 21:00:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.707107,0.707107,vila_militar,-22.861389,-43.411389,2023,9
533710,A621,2023-09-30 22:00:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.500000,0.866025,vila_militar,-22.861389,-43.411389,2023,9


# Radar Meteorológico do INEA

In [ ]:
path = 'curated/radar/inea/guaratiba'
table = dl.read_parquet_dataset(path)
table.schema

In [ ]:
df = table.to_pandas()
df